# Installation and Imports

In [1]:
from qiskit import QuantumCircuit
from mqt.qmap.na.zoned import ZonedNeutralAtomArchitecture
from mqt.qmap.na.zoned import RoutingAwareCompiler
from res_estimate_utils import remove_mid_circ_meas
from collections import defaultdict
from qiskit import transpile


# Implementation

## Define Circuit

In [3]:
with open("folded_cultivation_circ.txt", "r") as file:
    qasm_string = file.read()
qiskit_circuit = QuantumCircuit.from_qasm_str(qasm_string)

qiskit_circuit, meas_moves = remove_mid_circ_meas(qiskit_circuit, return_meas_reset_moves=True)

qiskit_counts = defaultdict(int)

for inst in qiskit_circuit.data:
     qiskit_counts[len(inst.qubits)] += 1
print(qiskit_counts)

defaultdict(<class 'int'>, {131: 64, 1: 68, 2: 144, 3: 8})


In [4]:
print(f"Number of qubits in circuit: {qiskit_circuit.num_qubits}")
depth =  qiskit_circuit.depth()
print(f"Circuit depth: {depth}")

Number of qubits in circuit: 131
Circuit depth: 51


## Compile circuits to native gate set

In [8]:
basis_gates = ['rx', 'rz', 'cz']
transpiled_circuit = transpile(qiskit_circuit, basis_gates=basis_gates)

## Define Architecture

In [9]:
arch = ZonedNeutralAtomArchitecture.from_json_string("""{
  "name": "Architecture with one entanglement and one storage zone",
  "operation_duration": {"rydberg_gate": 0.36, "single_qubit_gate": 52, "atom_transfer": 15},
  "operation_fidelity": {"rydberg_gate": 0.995, "single_qubit_gate": 0.9997, "atom_transfer": 0.999},
  "qubit_spec": {"T": 1.5e6},
  "storage_zones": [{
    "zone_id": 0,
    "slms": [{"id": 0, "site_separation": [3, 3], "r": 20, "c": 100, "location": [0, 0]}],
    "offset": [0, 0],
    "dimension": [297, 57]
  }],
  "entanglement_zones": [{
    "zone_id": 0,
    "slms": [
      {"id": 1, "site_separation": [12, 10], "r": 7, "c": 20, "location": [35, 67]},
      {"id": 2, "site_separation": [12, 10], "r": 7, "c": 20, "location": [37, 67]}
    ],
    "offset": [35, 67],
    "dimension": [230, 60]
  }],
  "aods": [{"id": 0, "site_separation": 2, "r": 100, "c": 100}],
  "rydberg_range": [[[30, 62], [270, 132]]]
}""")

## Compile for movements using mqt

In [11]:
depth =  transpiled_circuit.depth()
print(f"Circuit depth: {depth}")

Circuit depth: 245


In [12]:
from res_estimate_utils import calculate_movements_for_arch

compiler = RoutingAwareCompiler(arch)

compiler_moves = calculate_movements_for_arch(arch, transpiled_circuit , compiler)

In [13]:
from qiskit import QuantumCircuit

def remove_non_global_barriers(circuit: QuantumCircuit) -> QuantumCircuit:
    num_qubits = circuit.num_qubits
    new_circ = QuantumCircuit(num_qubits, circuit.num_clbits)
    for instr, qargs, cargs in circuit.data:
        if instr.name == "barrier":
            # Only keep if it's a global barrier
            if len(qargs) == num_qubits:
                new_circ.append(instr, qargs, cargs)
        else:
            new_circ.append(instr, qargs, cargs)
    return new_circ

# Usage:
# circ = remove_non_global_barriers(circ)

In [14]:
from mqt.core import load

circ = load(remove_non_global_barriers(transpiled_circuit ))
code = compiler.compile(circ)
print(code)

atom (0.000, 54.000) atom100
atom (0.000, 57.000) atom0
atom (3.000, 54.000) atom101
atom (3.000, 57.000) atom1
atom (6.000, 54.000) atom102
atom (6.000, 57.000) atom2
atom (9.000, 54.000) atom103
atom (9.000, 57.000) atom3
atom (12.000, 54.000) atom104
atom (12.000, 57.000) atom4
atom (15.000, 54.000) atom105
atom (15.000, 57.000) atom5
atom (18.000, 54.000) atom106
atom (18.000, 57.000) atom6
atom (21.000, 54.000) atom107
atom (21.000, 57.000) atom7
atom (24.000, 54.000) atom108
atom (24.000, 57.000) atom8
atom (27.000, 54.000) atom109
atom (27.000, 57.000) atom9
atom (30.000, 54.000) atom110
atom (30.000, 57.000) atom10
atom (33.000, 54.000) atom111
atom (33.000, 57.000) atom11
atom (36.000, 54.000) atom112
atom (36.000, 57.000) atom12
atom (39.000, 54.000) atom113
atom (39.000, 57.000) atom13
atom (42.000, 54.000) atom114
atom (42.000, 57.000) atom14
atom (45.000, 54.000) atom115
atom (45.000, 57.000) atom15
atom (48.000, 54.000) atom116
atom (48.000, 57.000) atom16
atom (51.000, 5

/tmp/ipykernel_1004453/1546189377.py:6: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, cargs in circuit.data:


In [15]:
print(f"Compiler predicted move: {compiler_moves}, (reset+meas)moves: {meas_moves}")
print(f"Total: {compiler_moves+meas_moves}(moves)")

Compiler predicted move: 541, (reset+meas)moves: 93
Total: 634(moves)
